# 03 — Neuroglancer Visualization

Interactive visualization of processed tissue sections using Neuroglancer.

**Three viewing modes:**
- **Option A** — OME-Zarr plate view (multi-tissue, one layer per child `.ome.zarr`)
- **Option B** — Precomputed volume (single `precomputed://` URL)
- **Option C** — Shareable state JSON files

**Prerequisites:** `pip install -e '.[visualization]'` (includes `neuroglancer`)

In [ ]:
from pathlib import Path

from wsi_pipeline.visualization import (
    open_neuroglancer_plate_view,
    open_neuroglancer_precomputed,
    stop_cors_server,
    RGB_SHADER_PLAIN,
    RGB_SHADER,
    R_SHADER,
    G_SHADER,
    B_SHADER,
)
from wsi_pipeline.neuroglancer_utils import (
    emit_ng_state_for_ngff_plate,
    emit_ng_state_for_precomputed_plate,
)

## Option A: View OME-Zarr Plate (Multi-Tissue)

`open_neuroglancer_plate_view()` finds all `*.ome.zarr` (or `*.zarr`) children
in a plate directory, reads physical pixel sizes from NGFF metadata, starts a
CORS + Range HTTP server, and launches Neuroglancer with one layer per tissue.

Set `visible_first_only=True` to avoid visual clutter — only the first layer is
visible by default; toggle others in the Neuroglancer sidebar.

In [ ]:
# --- Edit this path ---
plate_root = Path(
    "./data/output/per_tissue_ngff"  # ← Replace with the path to your OME-Zarr plate directory
).expanduser().resolve()

viewer, httpd = open_neuroglancer_plate_view(
    plate_root,
    mode="remote",
    http_host="localhost",
    http_port=8000,
    ng_host="localhost",
    ng_port=9999,
    shader=RGB_SHADER_PLAIN,
    visible_first_only=True,
)
# Paste the printed URL into Chrome or Firefox

In [ ]:
# Optional: embed the viewer directly in the notebook
from IPython.display import IFrame
IFrame(src=viewer.get_viewer_url(), width=1000, height=700)

## Option B: View Precomputed Volume

`open_neuroglancer_precomputed()` accepts a `precomputed://` URL.  When the
URL uses the `file://` scheme, it automatically starts a local HTTP server
and rewrites the URL so the Neuroglancer browser client can access the data.

Supported URL schemes:
- `precomputed://file:///ABS/PATH/volume`
- `precomputed://http(s)://host/path/to/volume`
- `precomputed://gs://bucket/path`
- `precomputed://s3://bucket/path`

In [ ]:
# Stop the previous server first
stop_cors_server(httpd)

viewer, httpd = open_neuroglancer_precomputed(
    "precomputed://file:///path/to/your/precomputed_plate",  # ← Replace with your precomputed volume path
    mode="remote",
    http_host="localhost",
    http_port=8000,
    ng_host="localhost",
    ng_port=9999,
)

## Option C: Generate Shareable State JSON

Write a Neuroglancer state JSON file that any Neuroglancer instance can load.
This is useful for sharing links with collaborators who have access to the
same HTTP-served data.

In [ ]:
# For NGFF plate
emit_ng_state_for_ngff_plate(
    plate_root=plate_root,
    base_http_url="http://localhost:8000/per_tissue_ngff",
    out_state_path=plate_root / "ng_state.json",
)

# For precomputed volume
# emit_ng_state_for_precomputed_plate(
#     precomputed_url="precomputed://http://localhost:8000/precomputed_plate",
#     out_state_path=Path("/data/out/precomputed_plate/ng_state.json"),
# )

## Shader Options

Available shader constants for RGB tissue images:

| Constant | Description |
|----------|-------------|
| `RGB_SHADER_PLAIN` | Basic `emitRGB(toNormalized(...))` |
| `RGB_SHADER` | Per-channel `#uicontrol invlerp` contrast sliders |
| `R_SHADER` | Red channel only (false-color) |
| `G_SHADER` | Green channel only (false-color) |
| `B_SHADER` | Blue channel only (false-color) |

Pass the desired shader via the `shader=` parameter.

In [ ]:
# Example: re-launch with the red-channel-only shader
stop_cors_server(httpd)

viewer, httpd = open_neuroglancer_plate_view(
    plate_root,
    mode="remote",
    http_host="localhost",
    http_port=8000,
    ng_host="localhost",
    ng_port=9999,
    shader=R_SHADER,
    visible_first_only=True,
)

## Cleanup

In [ ]:
stop_cors_server(httpd)